# CS-781 Phase 3

In [ ]:
# ---------------- CONFIG ----------------
TOKENIZER_PATH = "/kaggle/input/models/harsvij/cap-tok/pytorch/default/1/legal_bpe_v3.pkl"
CKPT_PATH = None            # None = auto-search /kaggle/input for a fusion*.pt checkpoint

FT_MODE = "full"            # "full" = all weights | "partial" = top N blocks + final norm + head
N_TOP_BLOCKS = 4            # used only when FT_MODE == "partial"

MAX_PREV, MAX_CURR, MAX_NEXT = 80, 192, 80     # token budgets per part
BATCH_SIZE = 32
EPOCHS = 5
LR_BACKBONE = 5e-5
LR_HEAD = 1e-3
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.1
GRAD_CLIP = 1.0
SEED = 42

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
OUT_DIR = "/kaggle/working"
RUN_TAG = f"phase3_fusion_{FT_MODE}"
UPLOAD_TO_HF = False        # needs a Kaggle secret; leave False unless you want it
TARGET_REPO = "hv-240436/cs781-submissions"

In [ ]:
import os, re, json, glob, time, copy, math, pickle, shutil, zipfile, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score
from huggingface_hub import hf_hub_download

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print("Device:", device, "|", torch.cuda.get_device_name(0) if USE_AMP else "cpu")

# Optional login through a Kaggle secret (never paste a token into the notebook)
HF_TOKEN = hf_token
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    HF_TOKEN = UserSecretsClient().get_secret("CS781_WRITE")
    login(token=HF_TOKEN)
    print("Hugging Face login OK")
except Exception as e:
    print("No HF login (fine if the dataset is public):", type(e).__name__)

## Tokenizer (copied unchanged from the Phase-1 notebook)

In [ ]:
import re, pickle


class LegalBPETokenizer:

    def __init__(self, model_path):

        with open(model_path, "rb") as f:
            model = pickle.load(f)

        self.vocab_size = model["vocab_size"]
        self.merges = model["merges"]                     # [((a, b), new_id), ...]
        self.lowercase = model.get("lowercase", False)
        self.special_tokens = model.get("special_tokens", [])
        self.pattern = re.compile(model["pattern"])

        # pair -> rank (lower rank = learned earlier = merged first)
        self.merge_rank = {
            tuple(pair): rank
            for rank, (pair, _) in enumerate(self.merges)
        }

        # id -> bytes table (O(1) decode)
        self.token_bytes = {i: bytes([i]) for i in range(256)}
        for (l, r), new_id in self.merges:
            self.token_bytes[new_id] = self.token_bytes[l] + self.token_bytes[r]

        base = 256 + len(self.merges)
        self.special_to_id = {t: base + i for i, t in enumerate(self.special_tokens)}
        self.id_to_special = {v: k for k, v in self.special_to_id.items()}
        self.eos_id = self.special_to_id.get("<|endoftext|>")

        self._special_re = (
            re.compile("(" + "|".join(re.escape(t) for t in self.special_tokens) + ")")
            if self.special_tokens else None
        )
        self._cache = {}

    def _encode_piece(self, piece_bytes):

        cached = self._cache.get(piece_bytes)
        if cached is not None:
            return cached

        ids = list(piece_bytes)

        while len(ids) > 1:
            best, best_rank = None, None
            for p in zip(ids, ids[1:]):
                r = self.merge_rank.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = p, r
            if best is None:
                break

            new_id = 256 + best_rank
            out, i = [], 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == best[0] and ids[i + 1] == best[1]:
                    out.append(new_id)
                    i += 2
                else:
                    out.append(ids[i])
                    i += 1
            ids = out

        if len(self._cache) < 500_000:
            self._cache[piece_bytes] = ids
        return ids

    def _encode_ordinary(self, text):

        if self.lowercase:
            text = text.lower()

        out = []
        for piece in self.pattern.findall(text):
            out.extend(self._encode_piece(piece.encode("utf-8")))
        return out

    def encode(self, text, add_eos=False):

        if self._special_re is not None:
            ids = []
            for part in self._special_re.split(text):
                if part in self.special_to_id:
                    ids.append(self.special_to_id[part])
                elif part:
                    ids.extend(self._encode_ordinary(part))
        else:
            ids = self._encode_ordinary(text)

        if add_eos:
            ids.append(self.eos_id)

        return ids

    def decode(self, ids):

        chunks = []
        for token_id in ids:
            if token_id in self.id_to_special:
                chunks.append(self.id_to_special[token_id].encode("utf-8"))
            else:
                chunks.append(self.token_bytes[token_id])

        return b"".join(chunks).decode("utf-8", errors="replace")

tokenizer = LegalBPETokenizer(TOKENIZER_PATH)
EOS_ID = tokenizer.eos_id
print("Vocab size:", tokenizer.vocab_size, "| eos id:", EOS_ID)

## Model code (copied unchanged from the Phase-1 notebook)

In [ ]:
%%writefile model_arch_fusion.py

"""RNN + Transformer fusion causal language model.

Block:  x -> x + Mix(norm1(x)) -> x + MLP(norm2(x))
Mix  =  causal self-attention (RoPE) in PARALLEL with a unidirectional GRU on the
        same normed input, merged by a learned gate: (1-g)*attn + g*rnn.
The GRU branch is present in every `rnn_every`-th block. Output head is tied.
"""

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class FusionConfig:

    vocab_size: int

    # defaults mirror the decoder-only run so the two are easy to compare
    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    d_ff: int = 2816

    max_seq_len: int = 1024

    dropout: float = 0.0          # < 1 epoch of data: dropout mostly slows learning

    # ---- fusion-specific fields ----
    rnn_hidden: int = 512         # GRU hidden size (projected back to d_model)
    rnn_every: int = 2            # GRU branch in blocks where i % rnn_every == 0; 0 = no RNN (ablation)
    rnn_fp32: bool = True         # run the recurrent branch in fp32 even under fp16 autocast
    rope_theta: float = 10000.0
    norm_eps: float = 1e-5


class RMSNorm(nn.Module):

    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        xf = x.float()
        out = xf * torch.rsqrt(xf.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (out * self.weight.float()).type_as(x)


def build_rope(head_dim: int, max_len: int, theta: float):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(max_len).float()
    freqs = torch.outer(t, inv_freq)                 # (L, hd/2)
    emb = torch.cat([freqs, freqs], dim=-1)          # (L, hd)
    return emb.cos(), emb.sin()


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    # x: (B, H, T, hd); cos/sin: (T, hd). Computed in fp32, returned in x.dtype
    xf = x.float()
    return (xf * cos + rotate_half(xf) * sin).type_as(x)


class CausalSelfAttention(nn.Module):

    def __init__(self, cfg: FusionConfig):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        assert self.head_dim % 2 == 0, "head_dim must be even for RoPE"
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.out = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.dropout = cfg.dropout

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)                                   # each (B, H, T, hd)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        y = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )
        y = y.transpose(1, 2).reshape(B, T, C)
        return self.out(y)


class RecurrentBranch(nn.Module):
    """Unidirectional GRU (strictly causal) followed by a projection to d_model."""

    def __init__(self, cfg: FusionConfig):
        super().__init__()
        self.fp32 = cfg.rnn_fp32
        self.gru = nn.GRU(cfg.d_model, cfg.rnn_hidden, num_layers=1,
                          batch_first=True, bidirectional=False)
        self.proj = nn.Linear(cfg.rnn_hidden, cfg.d_model, bias=False)

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        out_dtype = h.dtype
        if self.fp32:
            # keep the sequential recurrence out of fp16
            with torch.autocast(device_type=h.device.type, enabled=False):
                y, _ = self.gru(h.float())
        else:
            y, _ = self.gru(h)
        return self.proj(y.to(out_dtype))


class MLP(nn.Module):

    def __init__(self, cfg: FusionConfig):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.fc2 = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class FusionBlock(nn.Module):

    def __init__(self, cfg: FusionConfig, use_rnn: bool):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = CausalSelfAttention(cfg)
        self.rnn = RecurrentBranch(cfg) if use_rnn else None
        self.gate = nn.Linear(cfg.d_model, cfg.d_model, bias=True) if use_rnn else None
        self.norm2 = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.mlp = MLP(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin):
        h = self.norm1(x)
        mixed = self.attn(h, cos, sin)
        if self.rnn is not None:
            r = self.rnn(h)
            g = torch.sigmoid(self.gate(h))
            mixed = (1.0 - g) * mixed + g * r
        x = x + self.drop(mixed)
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x


class FusionLM(nn.Module):
    """Contract used by the rest of the notebook:

    forward(idx: LongTensor (B, T)) -> logits FloatTensor (B, T, vocab_size)
      * strictly causal: logits[:, t] depends only on idx[:, :t+1]
      * works for any 1 <= T <= cfg.max_seq_len
      * exposes self.cfg (needs .max_seq_len)
    """

    def __init__(self, cfg: FusionConfig):

        super().__init__()

        self.cfg = cfg

        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([
            FusionBlock(cfg, use_rnn=(cfg.rnn_every > 0 and i % cfg.rnn_every == 0))
            for i in range(cfg.n_layers)
        ])
        self.norm_f = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight             # weight tying

        cos, sin = build_rope(cfg.d_model // cfg.n_heads, cfg.max_seq_len, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        # GPT-2 style: scale the projections that write into the residual stream
        res_std = 0.02 / math.sqrt(2 * cfg.n_layers)
        for name, p in self.named_parameters():
            if name.endswith(("attn.out.weight", "mlp.fc2.weight", "rnn.proj.weight")):
                nn.init.normal_(p, mean=0.0, std=res_std)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:

        B, T = idx.shape
        assert 1 <= T <= self.cfg.max_seq_len, f"sequence length {T} outside [1, {self.cfg.max_seq_len}]"

        x = self.drop(self.tok_emb(idx))
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]

        for blk in self.blocks:
            x = blk(x, cos, sin)

        x = self.norm_f(x)
        return self.lm_head(x)

In [ ]:
import importlib, sys
sys.modules.pop("model_arch_fusion", None)
model_arch_fusion = importlib.import_module("model_arch_fusion")
FusionConfig, FusionLM = model_arch_fusion.FusionConfig, model_arch_fusion.FusionLM
print("Fusion model code loaded.")

## Load the Phase-1 checkpoint

In [ ]:
if CKPT_PATH is None:
    cands = [p for p in glob.glob("/kaggle/input/**/*.pt", recursive=True) if "fusion" in os.path.basename(p).lower()]
    assert cands, "No fusion*.pt under /kaggle/input: attach the checkpoint or set CKPT_PATH"
    CKPT_PATH = sorted(cands)[-1]
print("Checkpoint:", CKPT_PATH)

ckpt = torch.load(CKPT_PATH, map_location="cpu")
cfg = FusionConfig(**ckpt["config"])
assert cfg.vocab_size == tokenizer.vocab_size, "tokenizer / checkpoint vocab mismatch"
backbone = FusionLM(cfg)
backbone.load_state_dict(ckpt["model_state"])
print(f"Loaded step {ckpt.get('step')} | layers={cfg.n_layers} d_model={cfg.d_model} rnn_every={cfg.rnn_every} "
      f"| {sum(p.numel() for p in backbone.parameters())/1e6:.1f}M params")
del ckpt

## Data

In [ ]:
os.makedirs(f"{OUT_DIR}/data_p3", exist_ok=True)
p3_files = ["Phase_3_RR/Train/CL_train.json", "Phase_3_RR/Train/CL_dev.json",
            "Phase_3_RR/Train/IT_train.json", "Phase_3_RR/Train/IT_dev.json",
            "Phase_3_RR/Train/label_vocab.json",
            "Phase_3_RR/Test/CL_test.json", "Phase_3_RR/Test/IT_test.json"]
for f in p3_files:
    dest = f"{OUT_DIR}/data_p3/{os.path.basename(f)}"
    if not os.path.exists(dest):
        shutil.copyfile(hf_hub_download(HF_DATASET_REPO, f, repo_type="dataset", token=HF_TOKEN), dest)

def load(name): return json.load(open(f"{OUT_DIR}/data_p3/{name}.json", encoding="utf-8"))
label2id = load("label_vocab"); id2label = {v: k for k, v in label2id.items()}
NUM_LABELS = len(label2id)
cl_train, cl_dev, it_train, it_dev = load("CL_train"), load("CL_dev"), load("IT_train"), load("IT_dev")
cl_test, it_test = load("CL_test"), load("IT_test")

train_docs = cl_train + it_train          # training only
dev_docs = cl_dev + it_dev                # held out: never trained on
print("labels:", NUM_LABELS, "| train docs:", len(train_docs), "| dev docs:", len(dev_docs),
      "| test docs:", len(cl_test) + len(it_test))

In [ ]:
# One training/eval sample per sentence: [DOC_POS] [PREV] prev [CURR] sent [NEXT] next <eos>
MK_PREV = tokenizer.encode(" [PREV] ")
MK_CURR = tokenizer.encode(" [CURR] ")
MK_NEXT = tokenizer.encode(" [NEXT] ")
NONE_IDS = tokenizer.encode("NONE")
_pos_cache = {}
def pos_ids(pct):
    if pct not in _pos_cache: _pos_cache[pct] = tokenizer.encode(f"[DOC_POS: {pct}%]")
    return _pos_cache[pct]

def doc_items(docs):
    # train/dev: list of {"id","text":[sents],"labels":[...]} | test: {id: [sents]}
    if isinstance(docs, dict):
        return [{"id": k, "text": (v["text"] if isinstance(v, dict) else v), "labels": []} for k, v in docs.items()]
    return docs

def build_samples(docs, with_labels=True):
    samples = []
    for doc in doc_items(docs):
        sents, labels = doc["text"], doc.get("labels", [])
        n = len(sents)
        enc = [tokenizer.encode(s) for s in sents]            # tokenise each sentence once
        for i in range(n):
            pct = int(100 * i / max(1, n - 1))
            prev = enc[i - 1][-MAX_PREV:] if i > 0 else NONE_IDS          # keep the END of prev
            nxt = enc[i + 1][:MAX_NEXT] if i < n - 1 else NONE_IDS        # keep the START of next
            ids = pos_ids(pct) + MK_PREV + prev + MK_CURR + enc[i][:MAX_CURR] + MK_NEXT + nxt + [EOS_ID]
            s = {"doc_id": doc["id"], "sent_idx": i, "ids": ids}
            if with_labels: s["label"] = label2id[labels[i]]
            samples.append(s)
    return samples

train_samples = build_samples(train_docs)
dev_samples = build_samples(dev_docs)
print("train samples:", len(train_samples), "| dev samples:", len(dev_samples),
      "| max len:", max(len(s["ids"]) for s in train_samples))

# class weights from TRAIN labels only (inverse frequency, rarest class gets the largest weight, most common = 1.0)
counts = np.bincount([s["label"] for s in train_samples], minlength=NUM_LABELS).astype(float)
class_weights = torch.tensor(counts.max() / np.maximum(counts, 1), dtype=torch.float32, device=device)
print("class weights:", np.round(class_weights.cpu().numpy(), 2))

class SentDataset(Dataset):
    def __init__(self, samples): self.s = samples
    def __len__(self): return len(self.s)
    def __getitem__(self, i): return self.s[i]

def collate(batch):
    L = max(len(b["ids"]) for b in batch)
    ids = torch.zeros(len(batch), L, dtype=torch.long)            # right-pad; causal model never reads the padding
    lengths = torch.tensor([len(b["ids"]) for b in batch])
    for j, b in enumerate(batch): ids[j, :len(b["ids"])] = torch.tensor(b["ids"])
    out = {"ids": ids, "lengths": lengths}
    if "label" in batch[0]: out["label"] = torch.tensor([b["label"] for b in batch])
    out["doc_id"] = [b["doc_id"] for b in batch]; out["sent_idx"] = [b["sent_idx"] for b in batch]
    return out

train_loader = DataLoader(SentDataset(train_samples), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
dev_loader = DataLoader(SentDataset(dev_samples), batch_size=64, shuffle=False, collate_fn=collate)

## Classifier
Runs the Fusion blocks directly (skipping the LM head) and classifies from the hidden state of the last real token.

In [ ]:
class FusionSeqClassifier(nn.Module):
    def __init__(self, backbone, num_labels, dropout=0.1):
        super().__init__()
        self.bb = backbone
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(backbone.cfg.d_model, num_labels)
        nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)

    def forward(self, ids, lengths):
        B, T = ids.shape
        x = self.bb.tok_emb(ids)
        cos, sin = self.bb.rope_cos[:T], self.bb.rope_sin[:T]
        for blk in self.bb.blocks:
            x = blk(x, cos, sin)
        x = self.bb.norm_f(x)
        pooled = x[torch.arange(B, device=x.device), lengths.to(x.device) - 1]   # last real token
        return self.head(self.drop(pooled)).float()

model = FusionSeqClassifier(backbone, NUM_LABELS)

if FT_MODE == "partial":
    for p in model.bb.parameters(): p.requires_grad = False
    for blk in model.bb.blocks[-N_TOP_BLOCKS:]:
        for p in blk.parameters(): p.requires_grad = True
    for p in model.bb.norm_f.parameters(): p.requires_grad = True
else:
    assert FT_MODE == "full", "FT_MODE must be 'full' or 'partial'"

model.to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"FT_MODE={FT_MODE} | trainable {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

## Training (best epoch chosen on the held-out dev split)

In [ ]:
def evaluate(m, loader):
    m.eval(); preds, trues = [], []
    with torch.no_grad():
        for b in loader:
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                logits = m(b["ids"].to(device), b["lengths"])
            preds += logits.argmax(-1).cpu().tolist(); trues += b["label"].tolist()
    return np.array(trues), np.array(preds)

bb_params = [p for p in model.bb.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    [{"params": bb_params, "lr": LR_BACKBONE}, {"params": model.head.parameters(), "lr": LR_HEAD}],
    weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
warm = int(WARMUP_FRAC * total_steps)
lr_scale = lambda s: (s + 1) / max(1, warm) if s < warm else max(0.0, (total_steps - s) / max(1, total_steps - warm))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_scale)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
criterion = nn.CrossEntropyLoss(weight=class_weights)

best_f1, best_state, log = -1.0, None, []
for epoch in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); tot = 0.0
    for b in train_loader:
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(b["ids"].to(device), b["lengths"])
        loss = criterion(logits, b["label"].to(device))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        tot += loss.item()
    t, p = evaluate(model, dev_loader)
    mf1 = f1_score(t, p, average="macro", labels=list(range(NUM_LABELS)), zero_division=0)
    log.append({"epoch": epoch, "train_loss": tot / len(train_loader), "dev_macro_f1": mf1})
    print(f"Epoch {epoch}/{EPOCHS} | loss {tot/len(train_loader):.4f} | dev macro-F1 {mf1:.4f} | {time.time()-t0:.0f}s")
    if mf1 > best_f1:
        best_f1 = mf1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
json.dump(log, open(f"{OUT_DIR}/train_log_{RUN_TAG}.json", "w"))
print(f"Restored best epoch: dev macro-F1 = {best_f1:.4f}")

## Held-out dev report

In [ ]:
t, p = evaluate(model, dev_loader)
print(f"Dev (held out) macro-F1 {f1_score(t, p, average='macro', labels=list(range(NUM_LABELS)), zero_division=0):.4f} | "
      f"P {precision_score(t, p, average='macro', zero_division=0):.4f} | R {recall_score(t, p, average='macro', zero_division=0):.4f}")
for i in range(NUM_LABELS):
    print(f"  {i:2d} {id2label[i]:<28} F1 {f1_score(t == i, p == i, zero_division=0):.4f}  (support {int((t == i).sum())})")

## Test inference and submission zip

In [ ]:
test_docs = {**cl_test, **it_test}
test_samples = build_samples(test_docs, with_labels=False)
test_loader = DataLoader(SentDataset(test_samples), batch_size=64, shuffle=False, collate_fn=collate)
print("test docs:", len(test_docs), "| test sentences:", len(test_samples))

per_doc = {d: [] for d in test_docs}
model.eval()
with torch.no_grad():
    for b in test_loader:
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            pr = model(b["ids"].to(device), b["lengths"]).argmax(-1).cpu().tolist()
        for d, si, pi in zip(b["doc_id"], b["sent_idx"], pr):
            per_doc[d].append((si, id2label[pi]))

final = {}
for d, lst in per_doc.items():
    lst.sort(key=lambda x: x[0]); final[d] = [l for _, l in lst]
    v = test_docs[d]; n = len(v["text"] if isinstance(v, dict) else v)
    assert len(final[d]) == n, f"length mismatch in {d}: {len(final[d])} vs {n}"
print("length check passed for all test documents")

pred_path = f"{OUT_DIR}/prediction.json"
zip_path = f"{OUT_DIR}/submission_{RUN_TAG}.zip"
json.dump(final, open(pred_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname="prediction.json")
print("Created", zip_path, f"({os.path.getsize(zip_path)/1024:.1f} KB)", zipfile.ZipFile(zip_path).namelist())

if UPLOAD_TO_HF:
    from huggingface_hub import HfApi
    HfApi(token=HF_TOKEN).upload_file(path_or_fileobj=zip_path, path_in_repo=f"phase3/{os.path.basename(zip_path)}",
                                      repo_id=TARGET_REPO, repo_type="dataset")
    print("Uploaded to Hugging Face")